# Solve Skill — Notebook Control

ManiSkill 모션 플래닝으로 **액션을 처음부터 생성** (다운로드 아님). 결과는 액션 + env_states만
담긴 원본 궤적 (`obs_mode=none`, 이미지 없음). 이후 `generate`가 리플레이하며 RGB를 입힘.

mplib이 Linux 전용이라 풀이는 **WSL conda env**에서 headless로 돌고, 이 노트북(Windows)이 그걸 구동.

**전제조건**: WSL Ubuntu-22.04 + Miniconda `maniskill` env + `mesa-vulkan-drivers` (README 'WSL 환경 준비').

## Setup — import path

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print('project root:', PROJECT_ROOT)

## ① Function mode — PickCube 3개 액션 생성 (WSL 구동)

In [ ]:
from scripts.solve import run, SUPPORTED_TASKS
print('supported tasks:', SUPPORTED_TASKS)

traj = run(task='PickCube-v1', count=3)
print('solved raw trajectory:', traj)
print('exists:', traj.exists())

## ② 생성된 원본 궤적 구조 확인 — 액션 있음, 이미지 없음

In [ ]:
import h5py, json

with h5py.File(traj, 'r') as f:
    keys = list(f.keys())
    print('episodes:', len(keys), '->', keys)
    g = f[keys[0]]
    def walk(g, p=''):
        for k in g:
            it = g[k]
            if isinstance(it, h5py.Group): walk(it, p + k + '/')
            else: print(f'  {p+k:40s} {it.shape} {it.dtype}')
    walk(g)
meta = json.load(open(traj.with_suffix('.json')))
print('obs_mode:', meta['env_info']['env_kwargs'].get('obs_mode'),
      '| control_mode:', meta['env_info']['env_kwargs'].get('control_mode'))

## ③ 하이브리드 — solve 산출물을 generate로 리플레이해 RGB 입히기

`solve`(WSL) → `generate`(Windows) 전체 흐름. 우리가 만든 액션으로 학습용 데이터셋 완성.

In [ ]:
from scripts.generate import run as generate

dataset = generate(task='PickCube-v1', count=3, traj_path=str(traj))
print('RGB dataset:', dataset)

## ④ Subprocess mode — CLI 동등성 확인

In [ ]:
import subprocess, sys as _sys

result = subprocess.run(
    [_sys.executable, str(PROJECT_ROOT / 'scripts' / 'solve.py'),
     '--task', 'PickCube-v1', '--count', '2'],
    capture_output=True, text=True, encoding='utf-8', errors='replace',
)
print('returncode:', result.returncode)
print(result.stdout.strip())